In [ ]:
conda activate ucdogma_1.0

In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import scipy
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import sys
import scarches as sca
from scarches.dataset.trvae.data_handling import remove_sparsity
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
import seaborn as sns
import pycats
from pandas.api.types import CategoricalDtype
from lightning.pytorch import seed_everything

seed_everything(12345)
scvi.settings.seed = 12345
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
# color map
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

## Data Loading

- Global

In [ ]:
adata = sc.read("Reproducibility/Data/UC_TREKKER_RNA_ATAC_after_QC.h5ad")

In [ ]:
# Figure7B & S12A
n_epi = adata.obs['celltype'].nunique()
palette_epi = sns.color_palette("Set2", n_epi).as_hex()

palette_geno = ["#348ABD", "#A60628", "#7A68A6", "#467821",
                "#D55E00", "#CC79A7", "#56B4E9", "#009E73", "#F0E442", "#0072B2"]

sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=200)
sc.pl.umap(
    adata,
    color=['coarse_celltype'],
    palette=palette_epi,
    frameon=False,
    #size=2,
    legend_fontsize=4,
    show=False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/Figure7B.pdf", bbox_inches='tight')
plt.close()

sc.pl.umap(
    adata,
    color=['sample'],
    palette=palette_geno,
    frameon=False,
    size=2,
    legend_fontsize=4,
    show=False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS12A.pdf", bbox_inches='tight')
plt.close()


- Malignant subset

In [ ]:
adata = sc.read("Reproducibility/Data/UC_TREKKER_RNA_ATAC_Epithelial.h5ad")

In [ ]:
# Figure7D & S12B
palette_clone = ['#9fc3e7', '#f9d6a5', '#91cb98', '#f19b98', '#c5b7d9', '#dab89a', '#e6aece', '#b0dac7', 
                 '#d2bf1b', "#BFA4D6", '#f4b1ab', "#94b0cb", '#c9e3c1', '#dbc9e1', '#f5b081', "#acdad2"] 

n_celltype = adata.obs['epi_celltype'].nunique()
palette_celltype = sns.color_palette("husl", n_celltype).as_hex()


sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=200)
sc.pl.umap(
    adata,
    color=['clone'],
    palette=palette_clone,
    frameon=False,
    #size=2,
    legend_fontsize=4,
    show=False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/Figure7D.pdf", bbox_inches='tight')
plt.close()

sc.pl.umap(
    adata,
    color=['epi_celltype'],
    palette=palette_celltype,
    frameon=False,
    size=2,
    legend_fontsize=4,
    show=False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS12B.pdf", bbox_inches='tight')
plt.close()

sc.pl.umap(
    adata,
    color=['sample'],
    palette=palette_geno,
    frameon=False,
    size=2,
    legend_fontsize=4,
    show=False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS12B_sample.pdf", bbox_inches='tight')
plt.close()

In [ ]:
palette_clone = [
    "#E41A1C",  # Vivid Red
    "#377EB8",  # Vivid Blue
    "#4DAF4A",  # Vivid Green
    "#984EA3",  # Vivid Purple
    "#FF7F00",  # Vivid Orange
    "#FFFF33",  # Bright Yellow
    "#A65628",  # Brown
    "#F781BF",  # Hot Pink
    "#000000",  # Black
    "#1B9E77",  # Emerald
    "#D95F02",  # Orange-Deep
    "#7570B3",  # Indigo
    "#E7298A",  # Magenta
    "#66A61E",  # Lime-Green
    "#E6AB02",  # Amber
    "#A6761D",  # Ochre
    "#A50F15",  # Blood Red
    "#006D2C",  # Deep Forest Green
    "#08519C"   # Royal Blue
]

sc.pl.umap(
    adata,
    color=['clone2'],
    palette=palette_clone,
    frameon=False,
    #size=2,
    legend_fontsize=4,
    show=False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS13A.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Signature score
sig_df_1 = pd.read_csv("Reproducibility/Results/VISIONR/UC_TREKKER_Malignant_signature_score_hotspot.txt", sep='\t', index_col=0)
sig_df_2 = pd.read_csv("Reproducibility/Results/VISIONR/UC_TREKKER_Malignant_signature_score_literature.txt", sep='\t', index_col=0)

adata.obs = pd.merge(left=adata.obs, right=sig_df_1.loc[adata.obs_names,:], how='left', left_index=True, right_index=True)
adata.obs = pd.merge(left=adata.obs, right=sig_df_2.loc[adata.obs_names,:], how='left', left_index=True, right_index=True)

In [ ]:
sc.set_figure_params(figsize=(2, 2)) 
sc.set_figure_params(vector_friendly=True, dpi_save=50)
sc.pl.umap(
    adata,
    color=['Luminal','Squamous','Neural-like_progenitor'],
    color_map = "magma",    
    ncols=6,
    #vcenter = 0,
    vmax = 'p99',
    vmin = 'p1',
    use_raw=False,
    frameon=False,
    wspace=0.5,
    legend_fontsize=1.5,
    show = False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS12C.pdf", bbox_inches='tight')
plt.close()

In [ ]:
sc.set_figure_params(figsize=(2, 2)) 
sc.set_figure_params(vector_friendly=True, dpi_save=50)
sc.pl.umap(
    adata,
    color=['Hypoxia','Stress_Barkley'],
    color_map = "magma",    
    ncols=6,
#    vcenter = 0,
    vmax = 'p99',
    vmin = 'p1',
    use_raw=False,
    frameon=False,
    wspace=0.5,
    legend_fontsize=1.5,
    show = False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS13B.pdf", bbox_inches='tight')
plt.close()

- TF activity

In [ ]:
tmp_path = "Reproducibility/Results/LINGER/TREKKER/output/cell_population_TF_activity.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

cell_use = np.intersect1d(adata.obs_names, TF_activity.columns)
adata_tmp = adata[cell_use,:].copy()

# TF_activity
adata_TF = sc.AnnData(TF_activity.T.loc[adata_tmp.obs_names,:].copy(), obs=adata_tmp.obs)
sc.pp.scale(adata_TF, max_value=10)

# Leiden and UMAP by latent
adata_TF.obsm["MultiVI_latent"] = adata_tmp.obsm["MultiVI_latent"]
adata_TF.obsm["X_umap"] = adata_tmp.obsm["X_umap"]

In [ ]:
sc.set_figure_params(figsize=(2, 2)) 
sc.set_figure_params(vector_friendly=True, dpi_save=50)
sc.pl.umap(
    adata_TF,
    color=['FOXA1', 'NFE2L2'],
    color_map = solar_extra,    
    ncols=6,
#    vcenter = 0,
    vmax = 'p99',
    vmin = 'p1',
    use_raw=False,
    frameon=False,
    wspace=0.5,
    legend_fontsize=1.5,
    show = False
)
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS13C.pdf", bbox_inches='tight')
plt.close()

Concordance

In [ ]:

import pycats
from pandas.api.types import CategoricalDtype

df = adata.obs.copy()

celltype = {
    "SQM": ['P01_clone_1'],
    "LUM_1": ['P02_clone_1'],
    "LUM_2": ['P02_clone_2'],
    "LUM_3": ['P03_clone_1'],
    "LUM_4": ['P04_clone_1'],
    "LUM_5": ['P04_clone_2'],
    "LUM_6": ['P05_clone_1'],
    "LUM_7": ['P05_clone_2'],
    "LUM_8": ['P06_clone_1'],
    "LUM_9": ['P06_clone_2'],
    "LUM_10": ['P06_clone_3'],
    "LUM_11": ['P06_clone_4'],
    "LUM_12": ['P07_clone_1'],
    "LUM_13": ['P07_clone_2'],
    "NRP_1": ['P09_clone_1'],
    "NRP_2": ['P09_clone_2'],
}

df['clone_tmp'] = df["clone"].astype('category')
df['clone_tmp'] = pycats.cat_collapse(df["clone_tmp"], celltype)

accuracy = np.mean(df['epi_celltype'].astype(str) == df['clone_tmp'].astype(str))

print(accuracy)

In [ ]:
cat_type = CategoricalDtype(categories=['P01_clone_1', 'P02_clone_1', 'P02_clone_2', 'P03_clone_1', 
            'P04_clone_1', 'P04_clone_2', 
            'P05_clone_1', 'P05_clone_2', 
            'P06_clone_1', 'P06_clone_2', 'P06_clone_3', 'P06_clone_4', 
            'P07_clone_1', 'P07_clone_2', 
            'P09_clone_1', 'P09_clone_2'], ordered=True)
df['clone'] = df['clone'].astype(cat_type)

In [ ]:
cat_type = CategoricalDtype(categories=['SQM', 'LUM_1', 'LUM_2', 'LUM_3', 
            'LUM_4', 'LUM_5', 'LUM_6', 'LUM_7', 'LUM_8', 'LUM_9', 'LUM_10', 
            'LUM_11', 'LUM_12', 'LUM_13', 'NRP_1', 'NRP_2'], ordered=True)
df['epi_celltype'] = df['epi_celltype'].astype(cat_type)

In [ ]:
df = df.groupby(["epi_celltype", "clone"]).size().unstack(fill_value=0)
norm_df = df / df.sum(axis=0)

# Plot heatmap
plt.figure(figsize=(8, 8))
heatmap = plt.pcolor(norm_df, cmap="BuGn")  # Set colormap to 'Reds'
plt.colorbar(heatmap)  # Add color bar
plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns, rotation=90)
plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
plt.xlabel("clone")
plt.ylabel("celltype")
plt.title(f"celltype vs clone\nConcordance: {accuracy:.2f}")
plt.savefig(f"Reproducibility/Results/Plots/Slide-tags/FigureS12E.pdf", bbox_inches='tight')
plt.close()